<a href="https://colab.research.google.com/github/parthibanxd/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Lane:** `decline_recovery` — pages whose search position declines and either recover or do not.

This notebook uses `fact_content_daily_performance` from the FlyRank warehouse. Development is on the
mid-panel month `2026-03`; June 2026 is the sealed final month and is not used for label development.

**Run requirement:** save your `HF_TOKEN` as a Colab Secret. This notebook intentionally does not contain
a token or fabricated warehouse outputs.

In [3]:
# 0) Setup
!pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"

FEATURE_PATH = f"{WAREHOUSE}/fact_content_daily_performance/month={MID_MONTH}/*.parquet"
LABEL_PATH = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet"

print("Connected. Development month:", MID_MONTH)

Connected. Development month: 2026-03


## 1) The contract, in plain words

1. **What one row means for my lane:** one row is one pseudonymized content item for one pseudonymized client on one report date in `fact_content_daily_performance`. My lane groups those daily rows by `(client_hash_id, content_hash_id)` to detect a decline and then check whether it recovers.

2. **Table(s) I'll use:** `fact_content_daily_performance` is the primary source. `dim_content` and `dim_clients` are context/join tables only. IDs are used for grouping and joins, not as model features.

3. **Time window:** March 2026 is the feature/development window. For a March decision point, the recovery label looks forward into April 2026. June 2026 remains a sealed final month and is not used to develop the label.

4. **What I'd predict/rank:** a binary recovery proxy: after a meaningful position decline is detected, did the page return to within 20% of its trailing 7-day pre-decline position within the next 14 days? This is a proxy for whether the decline appears transient rather than persistent.

5. **What I deliberately exclude:** any metric from the future recovery window, the recovery label itself, and any product decision/output field. These are unavailable at the decision moment and would leak the answer.

## 2) Three verification queries

Run these against the March 2026 partition only.

In [4]:
# Query 1 — grain check
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_keys,
    COUNT(*) - COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS duplicate_key_rows
FROM read_parquet('{FEATURE_PATH}')
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys,duplicate_key_rows
0,9841378,9841378,0


In [5]:
# Query 2 — March slice row count and date span
slice_stats = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet('{FEATURE_PATH}')
""").df()

display(slice_stats)

,row_count,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [6]:
# Query 3 — availability, explicitly using IS TRUE
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS rows_before,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_after_gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_after_ga4_available
FROM read_parquet('{FEATURE_PATH}')
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_before,rows_after_gsc_available,rows_after_ga4_available
0,9841378,3611061,413966


### Reading the three checks

- Query 1 should show `duplicate_key_rows = 0`; otherwise the stated daily grain is wrong.
- Query 2 gives the actual March row count and date span.
- Query 3 deliberately uses `IS TRUE`, because the warehouse availability flags can be three-valued.

## 3) Five features + the leakage trap

The five features below are all available at the decision moment:

1. **current_position** — today's GSC average position.
2. **position_7d_avg** — trailing 7-day position average, using only rows up to today.
3. **position_change_7d** — today's position minus the position 7 days earlier.
4. **current_clicks** — today's GSC clicks.
5. **current_impressions** — today's GSC impressions.

The label is built separately from future April rows, so it is never included in the feature frame.

In [7]:
# Build the five-feature frame for March.
features = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_clicks,
        gsc_impressions
    FROM read_parquet('{FEATURE_PATH}')
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position IS NOT NULL
      AND gsc_avg_position > 0
)
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_avg_position AS current_position,
    AVG(gsc_avg_position) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS position_7d_avg,
    gsc_avg_position - LAG(gsc_avg_position, 7) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
    ) AS position_change_7d,
    gsc_clicks AS current_clicks,
    gsc_impressions AS current_impressions
FROM march
ORDER BY client_hash_id, content_hash_id, report_date
""").df()

display(features.head(10))
print("Feature rows:", len(features))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,current_position,position_7d_avg,position_change_7d,current_clicks,current_impressions
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,2026-03-26,9.000000,9.000000,NaN,0,1
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-01,6.166667,6.166667,NaN,0,6
2,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-02,12.888889,9.527778,NaN,0,9
3,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-03,26.000000,15.018519,NaN,0,5
4,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-04,12.625000,14.420139,NaN,0,8
5,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-05,22.714286,16.078968,NaN,1,7
6,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-06,12.333333,15.454696,NaN,0,12
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-07,10.400000,14.732596,NaN,0,5
8,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-08,10.083333,15.292120,3.916667,0,12
9,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-09,11.750000,15.129422,-1.138889,0,8


Feature rows: 3447872


### Why each feature is knowable

- **current_position:** available from today's search observation at the decision moment.
- **position_7d_avg:** calculated only from the current day and previous six observations.
- **position_change_7d:** compares today's position with an earlier observation, never a future row.
- **current_clicks:** today's observed GSC click count.
- **current_impressions:** today's observed GSC impression count.

In [8]:
# Build a recovery label for March decision points.
#
# A decline is detected when today's position is at least 20% worse than
# the trailing 7-day pre-decline average.
# Recovery = the best position observed in the next 14 days is within
# 20% of that pre-decline average.

label_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_avg_position,
        AVG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING
        ) AS pre_decline_7d_avg
    FROM read_parquet('{FEATURE_PATH}')
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position IS NOT NULL
      AND gsc_avg_position > 0
),
decision_points AS (
    SELECT *
    FROM march
    WHERE pre_decline_7d_avg IS NOT NULL
      AND gsc_avg_position >= pre_decline_7d_avg * 1.20
),
future AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_avg_position
    FROM read_parquet('{LABEL_PATH}')
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position IS NOT NULL
      AND gsc_avg_position > 0
)
SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.report_date,
    d.gsc_avg_position,
    d.pre_decline_7d_avg,
    CASE
        WHEN MIN(f.gsc_avg_position) <= d.pre_decline_7d_avg * 1.20
        THEN 1 ELSE 0
    END AS recovery_label
FROM decision_points d
LEFT JOIN future f
  ON f.client_hash_id = d.client_hash_id
 AND f.content_hash_id = d.content_hash_id
 AND f.report_date > d.report_date
 AND f.report_date <= d.report_date + INTERVAL 14 DAY
GROUP BY 1,2,3,4,5
ORDER BY 1,2,3
""").df()

display(label_frame.head(10))
print("Decision points:", len(label_frame))
print("Recovery positives:", int(label_frame["recovery_label"].sum()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_avg_position,pre_decline_7d_avg,recovery_label
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-02,12.888889,6.166667,0
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-03,26.000000,9.527778,0
2,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-05,22.714286,14.420139,0
3,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-14,18.666667,9.633191,0
4,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-18,14.222222,10.686752,0
5,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-20,15.384615,11.568234,0
6,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-21,17.000000,11.021225,0
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-22,15.666667,12.187892,1
8,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-23,19.285714,13.582336,1
9,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-24,21.000000,15.270981,1


Decision points: 954382
Recovery positives: 359556


In [9]:
# Join the label to the five-feature frame only after the features are created.
model_frame = features.merge(
    label_frame[["client_hash_id", "content_hash_id", "report_date", "recovery_label"]],
    on=["client_hash_id", "content_hash_id", "report_date"],
    how="inner"
)

print("Labeled feature rows:", len(model_frame))
display(model_frame.head(10))

Labeled feature rows: 954382


,client_hash_id,content_hash_id,report_date,current_position,position_7d_avg,position_change_7d,current_clicks,current_impressions,recovery_label
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-02,12.888889,9.527778,NaN,0,9,0
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-03,26.000000,15.018519,NaN,0,5,0
2,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-05,22.714286,16.078968,NaN,1,7,0
3,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-14,18.666667,10.923687,8.266667,0,9,0
4,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-18,14.222222,11.191819,5.333333,0,9,0
5,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-20,15.384615,12.113431,5.384615,0,13,0
6,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-21,17.000000,11.875336,-1.666667,0,11,0
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-22,15.666667,12.684860,5.666667,1,9,1
8,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-23,19.285714,14.397104,11.985714,0,14,1
9,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-24,21.000000,16.089412,11.846154,0,13,1


### Deliberate leakage trap

A label-derived feature should make the score look suspiciously good. We intentionally add the recovery
label itself as a fake feature, measure its correlation with the label, then remove it and report the honest
feature correlations.

The leaked feature is **not kept** in the final feature set.

In [10]:
# Deliberate leakage experiment: add ONE label-derived column on purpose.
leaky = model_frame[["recovery_label"]].copy()
leaky["leaky_feature"] = leaky["recovery_label"]

leak_score = leaky["leaky_feature"].corr(leaky["recovery_label"])
print(f"With deliberate label leakage, correlation = {leak_score:.3f}")

# Remove the leaked column.
honest_cols = [
    "current_position",
    "position_7d_avg",
    "position_change_7d",
    "current_clicks",
    "current_impressions",
]

honest = model_frame[honest_cols + ["recovery_label"]].dropna()

honest_scores = (
    honest[honest_cols]
    .corrwith(honest["recovery_label"])
    .abs()
    .sort_values(ascending=False)
)

print("\nHonest feature-label absolute correlations:")
display(honest_scores.to_frame("abs_correlation"))

final_feature_frame = honest[honest_cols + ["recovery_label"]].copy()
print("leaky_feature present in final frame:", "leaky_feature" in final_feature_frame.columns)

With deliberate label leakage, correlation = 1.000

Honest feature-label absolute correlations:


,abs_correlation
position_7d_avg,0.066706
current_position,0.059375
current_clicks,0.024775
position_change_7d,0.023424
current_impressions,0.022963


leaky_feature present in final frame: False


## 4) Limitation

**Named limitation:** this is an unbalanced daily panel. Client histories begin at different dates, and GSC
availability differs by client. Some content items therefore do not have enough prior observations to form
the trailing 7-day features or enough future observations to establish a 14-day recovery outcome. Missing
history is treated as unavailable evidence rather than as a zero signal.

## 5) Self-check

- [x] 5 plain-words contract answers written
- [x] 3 verification queries defined; outputs appear after running the notebook
- [x] Availability uses `IS TRUE`
- [x] Exactly 5 model features
- [x] Each feature has an explicit “knowable when?” explanation
- [x] Deliberate label leakage is added, its score is shown, then the leaked feature is removed
- [x] One named limitation
- [ ] **Run top-to-bottom in Colab, confirm outputs, save the executed notebook, and commit it to**
  `work/notebooks/w03_data_contract.ipynb`

### Final feature set

`current_position`, `position_7d_avg`, `position_change_7d`, `current_clicks`, `current_impressions`

### Final label

`recovery_label` = 1 when a detected March position decline recovers to within 20% of its pre-decline
7-day average within the following 14 days; otherwise 0.